# Global Carbon AOI Starter

**Title:** Global Carbon AOI Starter  
**Author:** Abhirsc and Codex  
**Project:** CarbonCalprac  
**Last updated:** 2026-03-23

This notebook is organized in step-by-step chunks for a first-time user.

Follow the notebook from top to bottom.

What this notebook does:

1. Installs the Python packages you need.
2. Sets a study area using a simple bounding box.
3. Queries open carbon datasets from the U.S. GHG Center.
4. Shows the AOI on a map with raster layers.
5. Prints a clear extent summary.
6. Tries to clip each dataset to the AOI and calculate simple statistics.
7. Produces first-pass insight text.

Default AOI: Australia.

If you see the map but not the final AOI statistics, do not worry. The notebook now prints separate extent details and status messages so you can tell exactly where things fail.


In [1]:
%pip install -q requests pystac-client pandas geopandas shapely folium matplotlib rasterio pyproj

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## Step 1. Import libraries and configure datasets

This cell defines the APIs and the default dataset stack:

- `ODIAC` for fossil-fuel CO2 emissions
- `MiCASA NPP` for land productivity related to carbon uptake
- `OCO-2 XCO2` for atmospheric carbon context


In [2]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Any

import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from folium import LayerControl
from IPython.display import Markdown, display
from pystac_client import Client
from pyproj import Transformer
from rasterio.mask import mask
from shapely.geometry import box, mapping

warnings.filterwarnings("ignore", category=UserWarning)

STAC_API_URL = "https://earth.gov/ghgcenter/api/stac"
RASTER_API_URL = "https://earth.gov/ghgcenter/api/raster"

COLLECTIONS = {
    "emissions": {
        "collection_id": "odiac-ffco2-monthgrid-v2023",
        "asset_name": "co2-emissions",
        "label": "ODIAC fossil-fuel CO2 emissions",
        "datetime": "2022-01-01/2022-12-31",
        "preferred_date": "2022-12",
        "colormap": "inferno",
    },
    "sequestration": {
        "collection_id": "micasa-carbonflux-monthgrid-v1",
        "asset_name": "npp",
        "label": "MiCASA net primary productivity",
        "datetime": "2022-01-01/2022-12-31",
        "preferred_date": "2022-12",
        "colormap": "viridis",
    },
    "atmospheric_context": {
        "collection_id": "oco2geos-co2-daygrid-v10r",
        "asset_name": "xco2",
        "label": "OCO-2 GEOS XCO2",
        "datetime": "2021-12-01/2021-12-31",
        "preferred_date": "2021-12",
        "colormap": "magma",
    },
}

catalog = Client.open(STAC_API_URL)

print("Configured datasets:")
for key, config in COLLECTIONS.items():
    print(f"- {key}: {config['collection_id']} | asset={config['asset_name']}")


/Users/abhirsc/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Configured datasets:
- emissions: odiac-ffco2-monthgrid-v2023 | asset=co2-emissions
- sequestration: micasa-carbonflux-monthgrid-v1 | asset=npp
- atmospheric_context: oco2geos-co2-daygrid-v10r | asset=xco2


## Step 2. Set the area of interest

For the first run, use the bounding box below. Later we can replace this with a GeoJSON polygon.

This step always prints an extent summary so you can confirm the AOI before any raster analysis starts.


In [3]:
AOI_NAME = "Australia"
AOI_BOUNDS = (112.0, -44.5, 154.5, -10.0)

aoi_geom = box(*AOI_BOUNDS)
aoi_gdf = gpd.GeoDataFrame({"name": [AOI_NAME]}, geometry=[aoi_geom], crs="EPSG:4326")
aoi_feature = mapping(aoi_geom)
aoi_area_km2 = float(aoi_gdf.to_crs(6933).area.iloc[0] / 1_000_000)

extent_summary_df = pd.DataFrame(
    [
        {
            "aoi_name": AOI_NAME,
            "min_lon": AOI_BOUNDS[0],
            "min_lat": AOI_BOUNDS[1],
            "max_lon": AOI_BOUNDS[2],
            "max_lat": AOI_BOUNDS[3],
            "approx_area_km2": round(aoi_area_km2, 0),
        }
    ]
)

display(extent_summary_df)
print(f"AOI '{AOI_NAME}' is ready.")


,aoi_name,min_lon,min_lat,max_lon,max_lat,approx_area_km2
0,Australia,112.0,-44.5,154.5,-10.0,15849337.0


AOI 'Australia' is ready.


## Step 3. Define helper functions

These functions search the STAC API, build map tiles, open clipped raster data, and summarize values.

Important change: the AOI statistics now use a direct raster read with clear error handling. If a dataset cannot be clipped, the notebook will show the error instead of failing silently.


In [4]:
@dataclass
class DatasetRun:
    name: str
    collection_id: str
    asset_name: str
    label: str
    item: Any
    item_date: str
    rescale_min: float
    rescale_max: float
    tile_url: str
    asset_href: str


def item_date_key(item: Any) -> str:
    for key in ("start_datetime", "datetime", "end_datetime"):
        value = item.properties.get(key)
        if value:
            return value[:10]
    return item.id


def search_items(collection_id: str, bbox: tuple[float, float, float, float], datetime_range: str, limit: int = 60) -> list[Any]:
    search = catalog.search(
        collections=[collection_id],
        bbox=list(bbox),
        datetime=datetime_range,
        limit=limit,
    )
    return list(search.items())


def choose_item(items: list[Any], preferred_date_prefix: str) -> Any:
    if not items:
        raise ValueError("No STAC items returned for this dataset and AOI.")

    for item in sorted(items, key=item_date_key):
        if item_date_key(item).startswith(preferred_date_prefix):
            return item

    return sorted(items, key=item_date_key)[-1]


def build_rescale(asset: Any) -> tuple[float, float]:
    raster_bands = asset.extra_fields.get("raster:bands", [{}])
    band = raster_bands[0] if raster_bands else {}
    stats = band.get("statistics", {})
    hist = band.get("histogram", {})

    if stats and "mean" in stats and "stddev" in stats:
        min_val = hist.get("min", stats.get("minimum", 0.0))
        max_val = stats["mean"] + (2 * stats["stddev"])
    else:
        min_val = hist.get("min", 0.0)
        max_val = hist.get("max", 1.0)

    return float(min_val), float(max_val)


def build_tile_url(collection_id: str, item_id: str, asset_name: str, color_map: str, rescale_min: float, rescale_max: float) -> str:
    response = requests.get(
        f"{RASTER_API_URL}/collections/{collection_id}/items/{item_id}/WebMercatorQuad/tilejson.json",
        params={
            "assets": asset_name,
            "color_formula": "gamma+r+1.05",
            "colormap_name": color_map.lower(),
            "rescale": f"{rescale_min},{rescale_max}",
        },
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["tiles"][0]


def normalize_asset_href(href: str) -> str:
    if href.startswith("s3://"):
        bucket, key = href[5:].split("/", 1)
        return f"https://{bucket}.s3.amazonaws.com/{key}"
    return href


def open_clipped_array(asset_href: str, geometry: dict[str, Any]) -> np.ndarray:
    with rasterio.open(asset_href) as src:
        clipped, _ = mask(src, [geometry], crop=True, filled=False)
        data = clipped[0].astype("float64")
        if np.ma.isMaskedArray(data):
            data = data.filled(np.nan)
        nodata = src.nodata
        if nodata is not None:
            data = np.where(data == nodata, np.nan, data)
    return data


def sample_value_at_lon_lat(asset_href: str, lon: float, lat: float) -> float:
    with rasterio.open(asset_href) as src:
        if src.crs is not None and str(src.crs) != "EPSG:4326":
            transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            x, y = transformer.transform(lon, lat)
        else:
            x, y = lon, lat

        sampled = next(src.sample([(x, y)]))[0]
        nodata = src.nodata
        if nodata is not None and sampled == nodata:
            return np.nan
        return float(sampled)


def summarize_array(data: np.ndarray) -> dict[str, float]:
    valid = data[np.isfinite(data)]
    if valid.size == 0:
        raise ValueError("The clipped raster contains no valid pixels inside the AOI.")

    return {
        "count": float(valid.size),
        "min": float(np.nanmin(valid)),
        "max": float(np.nanmax(valid)),
        "mean": float(np.nanmean(valid)),
        "sum": float(np.nansum(valid)),
    }


def hotspot_area_estimate_km2(data: np.ndarray, total_area_km2: float, quantile: float = 0.9) -> tuple[float, float]:
    valid = data[np.isfinite(data)]
    if valid.size == 0:
        return np.nan, np.nan

    threshold = float(np.nanquantile(valid, quantile))
    hotspot_fraction = float(np.mean(valid >= threshold))
    return threshold, hotspot_fraction * total_area_km2


def prepare_dataset_run(name: str, config: dict[str, Any], bbox: tuple[float, float, float, float]) -> DatasetRun:
    items = search_items(config["collection_id"], bbox, config["datetime"])
    item = choose_item(items, config["preferred_date"])
    asset = item.assets[config["asset_name"]]
    rescale_min, rescale_max = build_rescale(asset)
    tile_url = build_tile_url(
        collection_id=config["collection_id"],
        item_id=item.id,
        asset_name=config["asset_name"],
        color_map=config["colormap"],
        rescale_min=rescale_min,
        rescale_max=rescale_max,
    )
    asset_href = normalize_asset_href(asset.href)

    return DatasetRun(
        name=name,
        collection_id=config["collection_id"],
        asset_name=config["asset_name"],
        label=config["label"],
        item=item,
        item_date=item_date_key(item),
        rescale_min=rescale_min,
        rescale_max=rescale_max,
        tile_url=tile_url,
        asset_href=asset_href,
    )


## Step 4. Query the datasets for the current AOI

This step confirms that the STAC search is returning data before we try to clip anything.


In [5]:
dataset_runs = {}
availability_rows = []

for name, config in COLLECTIONS.items():
    items = search_items(config["collection_id"], AOI_BOUNDS, config["datetime"])
    availability_rows.append(
        {
            "dataset": name,
            "collection_id": config["collection_id"],
            "asset_name": config["asset_name"],
            "items_found": len(items),
            "first_item_date": item_date_key(sorted(items, key=item_date_key)[0]) if items else None,
            "last_item_date": item_date_key(sorted(items, key=item_date_key)[-1]) if items else None,
        }
    )
    dataset_runs[name] = prepare_dataset_run(name, config, AOI_BOUNDS)

availability_df = pd.DataFrame(availability_rows)
display(availability_df)


,dataset,collection_id,asset_name,items_found,first_item_date,last_item_date
0,emissions,odiac-ffco2-monthgrid-v2023,co2-emissions,12,2022-01-01,2022-12-01
1,sequestration,micasa-carbonflux-monthgrid-v1,npp,12,2022-01-01,2022-12-01
2,atmospheric_context,oco2geos-co2-daygrid-v10r,xco2,31,2021-12-01,2021-12-31


## Step 5. View the map and AOI extent

If the map renders, the data discovery and tile endpoint are working.

If you only want to confirm the AOI before moving on, this is enough.


In [6]:
center_lat = (AOI_BOUNDS[1] + AOI_BOUNDS[3]) / 2
center_lon = (AOI_BOUNDS[0] + AOI_BOUNDS[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=4, tiles="CartoDB positron")

for run in dataset_runs.values():
    folium.TileLayer(
        tiles=run.tile_url,
        attr=f"{run.label} via U.S. GHG Center",
        name=f"{run.label} ({run.item_date})",
        overlay=True,
        opacity=0.75,
    ).add_to(m)

folium.GeoJson(
    data=aoi_feature,
    name="AOI",
    style_function=lambda _: {"color": "black", "fillOpacity": 0.0, "weight": 2},
).add_to(m)

m.fit_bounds([[AOI_BOUNDS[1], AOI_BOUNDS[0]], [AOI_BOUNDS[3], AOI_BOUNDS[2]]])
LayerControl(collapsed=False).add_to(m)
m


## Step 6. Print a clear extent report

This is the new first-time-user checkpoint.

Even if later raster clipping fails, this section should still print exactly what extent the notebook is using and which item date was selected for each dataset.


In [7]:
extent_report_rows = []

for run in dataset_runs.values():
    extent_report_rows.append(
        {
            "aoi_name": AOI_NAME,
            "min_lon": AOI_BOUNDS[0],
            "min_lat": AOI_BOUNDS[1],
            "max_lon": AOI_BOUNDS[2],
            "max_lat": AOI_BOUNDS[3],
            "approx_area_km2": round(aoi_area_km2, 0),
            "dataset": run.label,
            "selected_item_date": run.item_date,
            "asset_name": run.asset_name,
        }
    )

extent_report_df = pd.DataFrame(extent_report_rows)
display(extent_report_df)


,aoi_name,min_lon,min_lat,max_lon,max_lat,approx_area_km2,dataset,selected_item_date,asset_name
0,Australia,112.0,-44.5,154.5,-10.0,15849337.0,ODIAC fossil-fuel CO2 emissions,2022-12-01,co2-emissions
1,Australia,112.0,-44.5,154.5,-10.0,15849337.0,MiCASA net primary productivity,2022-12-01,npp
2,Australia,112.0,-44.5,154.5,-10.0,15849337.0,OCO-2 GEOS XCO2,2021-12-01,xco2


## Step 7. Test that we are getting real data

This step reads one real raster value from the center of the AOI for each selected dataset.

If you see item IDs, asset URLs, and sample values here, then the notebook is reaching the actual remote assets and not just displaying map tiles.


In [8]:
live_check_rows = []

for name, run in dataset_runs.items():
    try:
        sample_value = sample_value_at_lon_lat(run.asset_href, center_lon, center_lat)
        live_check_rows.append(
            {
                "dataset": name,
                "label": run.label,
                "item_id": run.item.id,
                "item_date": run.item_date,
                "asset_name": run.asset_name,
                "sample_lon": round(center_lon, 4),
                "sample_lat": round(center_lat, 4),
                "sample_value": sample_value,
                "asset_href": run.asset_href,
                "status": "ok",
                "error": "",
            }
        )
    except Exception as exc:
        live_check_rows.append(
            {
                "dataset": name,
                "label": run.label,
                "item_id": run.item.id,
                "item_date": run.item_date,
                "asset_name": run.asset_name,
                "sample_lon": round(center_lon, 4),
                "sample_lat": round(center_lat, 4),
                "sample_value": np.nan,
                "asset_href": run.asset_href,
                "status": "failed",
                "error": str(exc),
            }
        )

live_check_df = pd.DataFrame(live_check_rows)
display(live_check_df)


,dataset,label,item_id,item_date,asset_name,sample_lon,sample_lat,sample_value,asset_href,status,error
0,emissions,ODIAC fossil-fuel CO2 emissions,odiac-ffco2-monthgrid-v2023-odiac2023_1km_excl...,2022-12-01,co2-emissions,133.25,-27.25,NaN,https://ghgc-data-store.s3.amazonaws.com/odiac...,failed,HTTP response code: 403
1,sequestration,MiCASA net primary productivity,micasa-carbonflux-monthgrid-v1-202212,2022-12-01,npp,133.25,-27.25,NaN,https://ghgc-data-store.s3.amazonaws.com/micas...,failed,HTTP response code: 403
2,atmospheric_context,OCO-2 GEOS XCO2,oco2geos-co2-daygrid-v10r-20211201,2021-12-01,xco2,133.25,-27.25,NaN,https://ghgc-data-store.s3.amazonaws.com/oco2g...,failed,HTTP response code: 403


## Step 8. Run AOI statistics

This is the part that can fail on some machines or providers.

The output table below now includes a `status` column and an `error` column so you can see what happened for each dataset.


In [9]:
stats_rows = []
clipped_arrays = {}

for name, run in dataset_runs.items():
    try:
        data = open_clipped_array(run.asset_href, aoi_feature)
        clipped_arrays[name] = data
        summary = summarize_array(data)
        stats_rows.append(
            {
                "dataset": name,
                "label": run.label,
                "item_date": run.item_date,
                "asset_name": run.asset_name,
                "status": "ok",
                "error": "",
                **summary,
            }
        )
    except Exception as exc:
        stats_rows.append(
            {
                "dataset": name,
                "label": run.label,
                "item_date": run.item_date,
                "asset_name": run.asset_name,
                "status": "failed",
                "error": str(exc),
                "count": np.nan,
                "min": np.nan,
                "max": np.nan,
                "mean": np.nan,
                "sum": np.nan,
            }
        )

stats_df = pd.DataFrame(stats_rows)
display(stats_df)


,dataset,label,item_date,asset_name,status,error,count,min,max,mean,sum
0,emissions,ODIAC fossil-fuel CO2 emissions,2022-12-01,co2-emissions,failed,HTTP response code: 403,NaN,NaN,NaN,NaN,NaN
1,sequestration,MiCASA net primary productivity,2022-12-01,npp,failed,HTTP response code: 403,NaN,NaN,NaN,NaN,NaN
2,atmospheric_context,OCO-2 GEOS XCO2,2021-12-01,xco2,failed,HTTP response code: 403,NaN,NaN,NaN,NaN,NaN


## Step 9. Create first-pass insights

This step only uses datasets that clipped successfully.

If a dataset failed in the previous step, the notebook will say so instead of crashing.


In [10]:
insight_lines = [
    f"AOI: {AOI_NAME}",
    f"Approx AOI area: {aoi_area_km2:,.0f} km^2",
]

for _, row in stats_df.iterrows():
    if row["status"] == "ok":
        insight_lines.append(
            f"{row['label']} on {row['item_date']}: mean={row['mean']:,.2f}, min={row['min']:,.2f}, max={row['max']:,.2f}, pixels={int(row['count'])}"
        )
    else:
        insight_lines.append(
            f"{row['label']} on {row['item_date']}: failed to clip. Error: {row['error']}"
        )

if "emissions" in clipped_arrays:
    threshold, hotspot_area_km2 = hotspot_area_estimate_km2(clipped_arrays["emissions"], aoi_area_km2, quantile=0.9)
    insight_lines.append(f"Approx emissions hotspot threshold (90th percentile): {threshold:,.2f}")
    insight_lines.append(f"Approx hotspot area above threshold: {hotspot_area_km2:,.0f} km^2")
else:
    insight_lines.append("Emissions hotspot area could not be estimated because the emissions raster did not clip successfully.")

display(Markdown("\n".join([f"- {line}" for line in insight_lines])))

print("Interpretation notes:")
print("- ODIAC is a modeled fossil-fuel CO2 emissions dataset, not a direct satellite observation.")
print("- MiCASA NPP is a productivity proxy related to carbon uptake, not an official sequestration audit.")
print("- OCO-2 XCO2 gives atmospheric carbon context, not an official inventory value.")
print("- Hotspot area is an AOI-relative estimate for study use.")


- AOI: Australia
- Approx AOI area: 15,849,337 km^2
- ODIAC fossil-fuel CO2 emissions on 2022-12-01: failed to clip. Error: HTTP response code: 403
- MiCASA net primary productivity on 2022-12-01: failed to clip. Error: HTTP response code: 403
- OCO-2 GEOS XCO2 on 2021-12-01: failed to clip. Error: HTTP response code: 403
- Emissions hotspot area could not be estimated because the emissions raster did not clip successfully.

Interpretation notes:
- ODIAC is a modeled fossil-fuel CO2 emissions dataset, not a direct satellite observation.
- MiCASA NPP is a productivity proxy related to carbon uptake, not an official sequestration audit.
- OCO-2 XCO2 gives atmospheric carbon context, not an official inventory value.
- Hotspot area is an AOI-relative estimate for study use.


## Next steps

Once this runs end to end, the next upgrades are:

1. Replace the bounding box with a GeoJSON polygon.
2. Add a time-series loop for one dataset at a time.
3. Add land-cover context.
4. Add a storage layer such as biomass or soil carbon.
